# Chandra2 local GPU smoke test

Notebook này chạy trực tiếp Chandra2 Hugging Face trên GPU của Google Colab, không gọi Datalab API hoặc vLLM server. Mặc định notebook chỉ xử lý trang đầu tiên với `batch_size=1` để kiểm tra môi trường trước khi chạy tài liệu đầy đủ.

## Kết nối từ VS Code

1. Cài extension **Colab** của Google trong VS Code.
2. Mở notebook này và chọn **Select Kernel → Colab → New Colab Server** (hoặc **Auto Connect**).
3. Đăng nhập tài khoản Google có Colab Pro và chọn runtime GPU. Ưu tiên L4 hoặc A100 vì Chandra2 tải model ở BF16.
4. Chạy các cell theo thứ tự. Filesystem Colab tách biệt với máy local; hãy upload tài liệu vào `/content` bằng mục **Colab Resources → Files**.

> Model được tải lần đầu từ Hugging Face và runtime Colab là tạm thời, nên việc cài dependency/tải model có thể phải lặp lại sau khi runtime bị reset.


In [1]:
import shutil
import subprocess
import sys

print(f"Python: {sys.version}")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Không tìm thấy GPU runtime. Hãy kết nối lại Colab với hardware accelerator là GPU.")
gpu_status = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,driver_version",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(f"GPU connected: {gpu_status.stdout.strip()}")


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
GPU connected: NVIDIA L4, 23034 MiB, 22564 MiB, 580.82.07


## 1. Cài Chandra2 với Hugging Face backend

Cell này cài package chính thức trực tiếp vào Python environment của kernel Colab. Nếu VS Code yêu cầu restart kernel sau khi cài, hãy restart rồi chạy lại từ cell cấu hình bên dưới.


In [2]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "chandra-ocr[hf]>=0.2,<0.3",
    ]
)


0

### T? ??ng materialize t?i li?u th?t tr?n Colab

PDF Ref.No.747_VietnamNationalUniversity,Hanoi.pdf ???c nh?ng v?o notebook. Ch?y cell d??i ??y s? t? gi?i m? file v?o /content; kh?ng m? file picker v? kh?ng c?n ch?n file th? c?ng. Khi d?ng kernel local, cell d?ng tr?c ti?p file trong data/raw/parser-smoke.


In [3]:
import base64
import hashlib
from pathlib import Path

LOCAL_REPO_ROOT = Path(r"D:\\AXIOM_DE-RD")
EXPECTED_INPUT_NAME = "Ref.No.747_VietnamNationalUniversity,Hanoi.pdf"
EXPECTED_INPUT_SHA256 = "f4b64c9b3f19e362917637a7a4333fd41f4385a595c9ff219b68352dc4221268"
LOCAL_INPUT_PATH = LOCAL_REPO_ROOT / "data/raw/parser-smoke" / EXPECTED_INPUT_NAME

EMBEDDED_INPUT_B64 = (
    "..."
)

try:
    import google.colab  # type: ignore[import-not-found]
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    UPLOADED_INPUT_PATH = Path("/content") / EXPECTED_INPUT_NAME
    embedded_bytes = base64.b64decode(EMBEDDED_INPUT_B64)
    embedded_sha256 = hashlib.sha256(embedded_bytes).hexdigest()
    if embedded_sha256 != EXPECTED_INPUT_SHA256:
        raise RuntimeError("Embedded PDF checksum mismatch.")
    if not UPLOADED_INPUT_PATH.is_file() or hashlib.sha256(UPLOADED_INPUT_PATH.read_bytes()).hexdigest() != EXPECTED_INPUT_SHA256:
        UPLOADED_INPUT_PATH.write_bytes(embedded_bytes)
        print(f"Materialized embedded input on Colab: {UPLOADED_INPUT_PATH}")
    else:
        print(f"Reusing existing Colab input: {UPLOADED_INPUT_PATH}")
else:
    if not LOCAL_INPUT_PATH.is_file():
        raise FileNotFoundError(LOCAL_INPUT_PATH)
    UPLOADED_INPUT_PATH = LOCAL_INPUT_PATH

actual_sha256 = hashlib.sha256(UPLOADED_INPUT_PATH.read_bytes()).hexdigest()
if actual_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError(f"Input checksum mismatch: {actual_sha256}")
print(f"Input: {UPLOADED_INPUT_PATH} ({UPLOADED_INPUT_PATH.stat().st_size:,} bytes)")


Materialized embedded input on Colab: /content/Ref.No.747_VietnamNationalUniversity,Hanoi.pdf
Input: /content/Ref.No.747_VietnamNationalUniversity,Hanoi.pdf (298,163 bytes)


## 2. Cấu hình model và input

Các biến môi trường phải được đặt **trước lần import đầu tiên của `chandra`**, vì package đọc chúng khi khởi tạo settings. Upload file rồi sửa `INPUT_PATH` nếu tên file khác.


In [4]:
import os
import re
from pathlib import Path

MODEL_CHECKPOINT = "datalab-to/chandra-ocr-2"
INPUT_PATH = Path(globals().get("UPLOADED_INPUT_PATH", "/content/input.pdf"))
if IN_COLAB:
    COLAB_REPO_ROOT = Path("/content/AXIOM_DE-RD")
    WORK_ROOT = (COLAB_REPO_ROOT if COLAB_REPO_ROOT.is_dir() else Path("/content")) / "data/work"
else:
    WORK_ROOT = LOCAL_REPO_ROOT / "data/work"
SAFE_INPUT_STEM = re.sub(r"[^A-Za-z0-9._-]+", "-", INPUT_PATH.stem).strip("-.") or "document"
OUTPUT_DIR = WORK_ROOT / "chandra2" / SAFE_INPUT_STEM
MAX_PAGES = 2
BATCH_SIZE = 1
MAX_OUTPUT_TOKENS = 4096  # Tăng lên 12384 sau khi smoke test ổn định.
INCLUDE_IMAGES = True
INCLUDE_HEADERS_FOOTERS = True

os.environ["MODEL_CHECKPOINT"] = MODEL_CHECKPOINT
os.environ["TORCH_DEVICE"] = "cuda"  # Buộc model chạy trên GPU thật.
os.environ.pop("TORCH_ATTN", None)

if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy {INPUT_PATH}. Upload file vào /content rồi cập nhật INPUT_PATH."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Kiểm tra CUDA và tải model

`InferenceManager(method="hf")` sẽ tải checkpoint trong lần chạy đầu tiên. Nếu GPU không hỗ trợ BF16 hoặc hết VRAM, hãy đổi runtime Colab sang L4/A100 thay vì bỏ `TORCH_DEVICE=cuda`; bỏ biến này có thể khiến model offload sang CPU và không còn là GPU smoke test thuần túy.


In [5]:
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch không nhận CUDA trong kernel hiện tại.")

gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / (1024 ** 3)
print(f"GPU: {gpu.name} ({gpu_memory_gib:.1f} GiB)")
print(f"CUDA: {torch.version.cuda}")
print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
if not torch.cuda.is_bf16_supported():
    print("WARNING: GPU này không hỗ trợ BF16 tốt; hãy ưu tiên Colab L4 hoặc A100.")

from chandra.input import load_file
from chandra.model import InferenceManager
from chandra.model.schema import BatchInputItem

pages = load_file(str(INPUT_PATH), {})
if not pages:
    raise RuntimeError("Chandra2 không đọc được trang nào từ input.")

print(f"Loaded pages: {len(pages)}")
if "manager" not in globals():
    manager = InferenceManager(method="hf")
    print("Chandra2 HF model loaded.")
else:
    print("Reusing the Chandra2 model already loaded in this runtime.")
subprocess.run(["nvidia-smi"], check=True)


GPU: NVIDIA L4 (22.0 GiB)
CUDA: 12.8
BF16 supported: True
Loaded pages: 2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.6GB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 20.0MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Chandra2 HF model loaded.


CompletedProcess(args=['nvidia-smi'], returncode=0)

## 4. Chạy OCR và lưu kết quả

Cấu hình hiện chạy cả 2 trang của `Ref.No.747_VietnamNationalUniversity,Hanoi.pdf`. Giữ `BATCH_SIZE=1` để giảm áp lực VRAM; tăng `MAX_OUTPUT_TOKENS` nếu một trang thật bị cắt output.


In [6]:
import json
import time
from collections import Counter

selected_pages = pages[: min(MAX_PAGES, len(pages))]
results = []
started = time.monotonic()

for start in range(0, len(selected_pages), BATCH_SIZE):
    page_batch = selected_pages[start : start + BATCH_SIZE]
    batch = [
        BatchInputItem(image=image, prompt_type="ocr_layout")
        for image in page_batch
    ]
    with torch.inference_mode():
        generated = list(
            manager.generate(
                batch,
                include_images=INCLUDE_IMAGES,
                include_headers_footers=INCLUDE_HEADERS_FOOTERS,
                max_output_tokens=MAX_OUTPUT_TOKENS,
            )
        )
    if len(generated) != len(batch):
        raise RuntimeError("Số output của Chandra2 không khớp số trang input.")
    failed = [start + i + 1 for i, item in enumerate(generated) if item.error]
    if failed:
        raise RuntimeError(f"Chandra2 lỗi tại các trang: {failed}")
    results.extend(generated)

latency_seconds = round(time.monotonic() - started, 3)
markdown = "\n\n".join(result.markdown or "" for result in results)
clean_html = "\n\n".join(
    f'<section data-page-number="{page_number}">\n{result.html or ""}\n</section>'
    for page_number, result in enumerate(results, start=1)
)
token_count = sum(int(result.token_count or 0) for result in results)

markdown_path = OUTPUT_DIR / "result.md"
html_path = OUTPUT_DIR / "result.html"
chunks_path = OUTPUT_DIR / "chunks.json"
metadata_path = OUTPUT_DIR / "metadata.json"
pages_dir = OUTPUT_DIR / "pages"
pages_dir.mkdir(parents=True, exist_ok=True)

markdown_path.write_text(markdown, encoding="utf-8")
html_path.write_text(clean_html, encoding="utf-8")

document_pages = []
label_counts = Counter()
for page_number, result in enumerate(results, start=1):
    page_prefix = f"page_{page_number:04d}"
    raw_path = pages_dir / f"{page_prefix}.raw.html"
    clean_path = pages_dir / f"{page_prefix}.clean.html"
    page_chunks_path = pages_dir / f"{page_prefix}.chunks.json"

    blocks = list(result.chunks or [])
    label_counts.update(block.get("label", "block") for block in blocks)
    page_payload = {
        "page_number": page_number,
        "page_box": list(result.page_box or []),
        "blocks": blocks,
    }

    raw_path.write_text(result.raw or "", encoding="utf-8")
    clean_path.write_text(result.html or "", encoding="utf-8")
    image_files = []
    for image_name, image in (result.images or {}).items():
        image_path = OUTPUT_DIR / Path(image_name).name
        image.save(image_path)
        image_files.append(image_path.name)
    page_payload["image_files"] = image_files
    page_chunks_path.write_text(
        json.dumps(page_payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    document_pages.append(page_payload)

chunks_path.write_text(
    json.dumps(
        {"input_path": str(INPUT_PATH), "pages": document_pages},
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
metadata_path.write_text(
    json.dumps(
        {
            "parser": "chandra2",
            "method": "hf",
            "model_checkpoint": MODEL_CHECKPOINT,
            "torch_device": "cuda",
            "gpu_name": gpu.name,
            "input_path": str(INPUT_PATH),
            "page_count": len(results),
            "token_count": token_count,
            "latency_seconds": latency_seconds,
            "label_counts": dict(sorted(label_counts.items())),
        },
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

print(f"Processed {len(results)} page(s), {token_count} tokens in {latency_seconds}s")
print(f"Markdown: {markdown_path}")
print(f"Clean HTML: {html_path}")
print(f"Layout chunks: {chunks_path}")
print(f"Per-page raw/clean/chunks: {pages_dir}")
print(f"Metadata: {metadata_path}")
print(f"Labels: {dict(sorted(label_counts.items()))}")
print("\nMarkdown preview:\n")
print(markdown[:3000])
subprocess.run(["nvidia-smi"], check=True)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Processed 2 page(s), 2062 tokens in 132.9s
Markdown: /content/data/work/chandra2/Ref.No.747_VietnamNationalUniversity-Hanoi/result.md
Clean HTML: /content/data/work/chandra2/Ref.No.747_VietnamNationalUniversity-Hanoi/result.html
Layout chunks: /content/data/work/chandra2/Ref.No.747_VietnamNationalUniversity-Hanoi/chunks.json
Per-page raw/clean/chunks: /content/data/work/chandra2/Ref.No.747_VietnamNationalUniversity-Hanoi/pages
Metadata: /content/data/work/chandra2/Ref.No.747_VietnamNationalUniversity-Hanoi/metadata.json
Labels: {'Caption': 1, 'Image': 1, 'List-Group': 2, 'Page-Header': 2, 'Section-Header': 3, 'Text': 26}

Markdown preview:

![ASEAN University Network logo](4ee14fc0ae64fcc5ff5c721ad736d49c_1_img.webp)

**ASEAN  
University  
Network**

17<sup>th</sup> Floor, Chaloem Rajakumari 60 (Chamchuri 10) Building,  
Chulalongkorn University, Phayathai Road,  
Bangkok 10330, Thailand  
Tel: (66 2) 2153640 / 2153642 / 2183256 / 2183258  
Fax: (66 2) 2168808  
Website: [www.aunsec.o

CompletedProcess(args=['nvidia-smi'], returncode=0)

## 5. Examine từng representation trước khi dựng cây

- `pages/page_XXXX.raw.html`: output gốc của model, còn các `div[data-label][data-bbox]`.
- `result.html`: HTML đã làm sạch để render; không còn metadata layout ở wrapper.
- `result.md`: thuận tiện cho đọc và RAG, nhưng đã mất bbox và phần lớn cấu trúc layout.
- `chunks.json`: block phẳng theo từng trang gồm `label`, bbox pixel và `content`; đây là candidate tốt nhất để dựng cây.

Cell dưới chỉ hiển thị để so sánh, chưa áp dụng luật suy luận `Section/Subsection` nên chưa tạo cây giả.


In [8]:
import shutil
from collections import Counter

from IPython.display import HTML, Markdown, display

EXAMINE_PAGE = 1
page_result = results[EXAMINE_PAGE - 1]
blocks = list(page_result.chunks or [])

print(f"Page {EXAMINE_PAGE}: {len(blocks)} blocks")
print("Label counts:", dict(Counter(block.get("label", "block") for block in blocks)))
print("\nFirst blocks (label | bbox | content preview):")
for index, block in enumerate(blocks[:10], start=1):
    content_preview = " ".join(str(block.get("content", "")).split())[:180]
    print(f"{index:02d}. {block.get('label')} | {block.get('bbox')} | {content_preview}")

print("\nRAW layout HTML preview:\n")
print((page_result.raw or "")[:4000])

print("\nRendered clean HTML:")
display(HTML(page_result.html or "<em>empty</em>"))
print("Rendered Markdown:")
display(Markdown(page_result.markdown or "_empty_"))

archive_base = Path("/content") / SAFE_INPUT_STEM if IN_COLAB else OUTPUT_DIR
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR)
print(f"Archive: {archive_path}")
if IN_COLAB:
    print("Trong VS Code: mở Colab activity bar > Contents, refresh /content,")
    print("right-click file ZIP ở trên > Download. Sau đó chạy scripts/import_chandra2_colab_output.cmd.")


Page 1: 18 blocks
Label counts: {'Image': 1, 'Page-Header': 2, 'Text': 12, 'Section-Header': 1, 'List-Group': 1, 'Caption': 1}

First blocks (label | bbox | content preview):
01. Image | [59, 41, 204, 190] | <img alt="ASEAN University Network logo"/>
02. Page-Header | [208, 56, 384, 170] | <p><b>ASEAN<br/>University<br/>Network</b></p>
03. Page-Header | [400, 41, 1004, 198] | <p>17<sup>th</sup> Floor, Chaloem Rajakumari 60 (Chamchuri 10) Building,<br/>Chulalongkorn University, Phayathai Road,<br/>Bangkok 10330, Thailand<br/>Tel: (66 2) 2153640 / 2153642
04. Text | [161, 220, 382, 250] | <p>Ref. No. 747/2026</p>
05. Text | [161, 291, 322, 324] | <p>18 May 2026</p>
06. Section-Header | [279, 361, 1253, 436] | <p><b>Subject: Invitation to the 21<sup>st</sup> ASEAN and 11<sup>th</sup> ASEAN+3 Youth Cultural Forum<br/>on 28 July - 2 August 2026 at Guizhou University, Guiyang, China</b></p>
07. Text | [159, 473, 637, 508] | <p>Dear Assoc. Prof. Dr. Hoang Minh Son,</p>
08. Text | [159, 546, 1

Rendered Markdown:


![ASEAN University Network logo](4ee14fc0ae64fcc5ff5c721ad736d49c_1_img.webp)

**ASEAN  
University  
Network**

17<sup>th</sup> Floor, Chaloem Rajakumari 60 (Chamchuri 10) Building,  
Chulalongkorn University, Phayathai Road,  
Bangkok 10330, Thailand  
Tel: (66 2) 2153640 / 2153642 / 2183256 / 2183258  
Fax: (66 2) 2168808  
Website: [www.aunsec.org](http://www.aunsec.org)

Ref. No. 747/2026

18 May 2026

**Subject: Invitation to the 21<sup>st</sup> ASEAN and 11<sup>th</sup> ASEAN+3 Youth Cultural Forum  
on 28 July - 2 August 2026 at Guizhou University, Guiyang, China**

Dear Assoc. Prof. Dr. Hoang Minh Son,

The ASEAN University Network (AUN) Secretariat, in collaboration with Guizhou University, is pleased to announce the organization of **the 21<sup>st</sup> ASEAN and 11<sup>th</sup> ASEAN+3 Youth Cultural Forum**, which will take place **from 28 July - 2 August 2026 at Guizhou University, Guiyang, People's Republic of China**.

As one of the flagship youth platforms of AUN, the ASEAN and ASEAN+3 Youth Cultural Forum is organized annually in partnership with its member universities to strengthen their cooperation through deep understanding of the cultural diversity of ASEAN and ASEAN+3 region. The forum provides a rich venue for youths endowed with talents in performance arts to showcase the beauty of ASEAN arts and culture through collaborative performances while preserving the region's traditional heritage.

In this regard, the AUN Secretariat and Guizhou University are honored to invite 4 students and 1 faculty member from your institution to participate in the aforementioned event. Following the standard cost-sharing practice, a participant will be responsible for their international travel expenses and a registration fee of USD 100 per person. The host university is responsible for organizing costs including accommodation, meals, local transportation, and venue.

Please take note of the general information of the 21<sup>st</sup> ASEAN and 11<sup>th</sup> ASEAN+3 Youth Cultural Forum below:

- • Arrival Date: 28 July 2026
- • Event Dates: 28 July (evening) - 2 August 2026
- • Departure Date: 2 August 2026

To confirm their participation, please complete the registration form at your earliest convenience, preferably by **26 June 2026**. Please do not hesitate to reach out to the AUN Secretariat in case of any further clarification of the event via our contact person.

We highly appreciate your continued support and look forward to receiving your positive response in due course.

Yours sincerely,

Assoc. Prof. Dr. Thanapan Laiprakobsup  
AUN Executive Director

Enclosures.../

Archive: /content/Ref.No.747_VietnamNationalUniversity-Hanoi.zip
Trong VS Code: mở Colab activity bar > Contents, refresh /content,
right-click file ZIP ở trên > Download. Sau đó chạy scripts/import_chandra2_colab_output.cmd.


In [9]:
!ls -lh /content/*.zip

-rw-r--r-- 1 root root 17K Jul 22 05:00 /content/Ref.No.747_VietnamNationalUniversity-Hanoi.zip


## 6. Bước tiếp theo: kiểm tra tích hợp AXIOM pipeline

Notebook này chủ ý kiểm tra Chandra2 trực tiếp trước. Khi smoke test thành công, clone branch dự án vào `/content/AXIOM_DE-RD`, chạy `pip install -e ".[chandra2-local]"`, đổi `parsing.document.provider` thành `chandra2`, `parsing.chandra2.method` thành `hf`, rồi gọi:

```python
from src.pipeline import run_pipeline
state = run_pipeline("configs/pipeline.colab.yaml")
assert not state.errors, state.errors
```

Không chỉ nhìn thông báo `completed`: luôn kiểm tra `state.errors` và metadata phải có `method: hf`.
